# Cleaning Data
## Oasis Infobyte - Data Analytics Internship (Task 3)

**Project:** Cleaning Data  
**Dataset:** Hotel Booking Dataset (Synthetic)  
**Goal:** Take a raw dataset, clean it thoroughly, document every preprocessing step, and generate a cleaned dataset ready for analysis or machine learning.

---

### Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Initial Dataset Inspection](#3-initial-dataset-inspection)
4. [Missing Value Handling](#4-missing-value-handling)
5. [Duplicate Removal](#5-duplicate-removal)
6. [Standardize Text Data](#6-standardize-text-data)
7. [Data Type Conversion](#7-data-type-conversion)
8. [Outlier Detection](#8-outlier-detection)
9. [Invalid Data Handling](#9-invalid-data-handling)
10. [Rename Columns](#10-rename-columns)
11. [Final Dataset Validation](#11-final-dataset-validation)
12. [Save Clean Dataset](#12-save-clean-dataset)
13. [Final Summary](#13-final-summary)


## 1. Import Libraries

We use the following libraries:
- **pandas** for data manipulation
- **numpy** for numerical operations
- **matplotlib** and **seaborn** for visualizations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")


## 2. Load Dataset

Load the raw `dataset.csv` file and display its basic structure.


In [ ]:
PROJECT_DIR = r"D:\roopank\coding\New folder (2)\OIBSIP\DataAnalytics-L3-CleaningData"
DATASET_PATH = os.path.join(PROJECT_DIR, "dataset.csv")
IMAGES_DIR = os.path.join(PROJECT_DIR, "images")

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.\n")

print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print(f"\nDataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nColumn names:\n{list(df.columns)}")


## 3. Initial Dataset Inspection

We inspect data types, missing values, duplicates, and memory usage before cleaning.


In [ ]:
print("Dataset Info:")
print(df.info())

print("\nStatistical Summary (numerical columns):")
display(df.describe())

print("\nMissing values per column:")
missing_counts = df.isnull().sum()
display(missing_counts[missing_counts > 0])

print(f"\nTotal duplicated rows: {df.duplicated().sum()}")

print("\nData types:")
display(df.dtypes)

mem_usage = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory usage: {mem_usage:.2f} MB")

# Save before visualizations
os.makedirs(IMAGES_DIR, exist_ok=True)

plt.figure(figsize=(14, 8))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap (Before Cleaning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "missing_values_before.png"), dpi=150, bbox_inches='tight')
plt.close()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
plot_df = df[numeric_cols].dropna()
plt.figure(figsize=(14, 8))
plot_df.boxplot()
plt.xticks(rotation=45)
plt.title('Boxplot of Numeric Features (Before Cleaning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "boxplot_before.png"), dpi=150, bbox_inches='tight')
plt.close()

print("\nSaved visualizations: missing_values_before.png, boxplot_before.png")


## 4. Missing Value Handling

We apply different imputation strategies based on column semantics:
- **Numerical (Age, ADR):** Median imputation (robust to outliers).
- **Categorical (Country, Gender):** Mode imputation.
- **Points/Requests (Loyalty_Points, Special_Requests):** Fill with 0.
- **Identifiers (Agent_ID):** Fill with 'Unknown'.
- **Critical columns:** Drop rows where core identifiers are missing.


In [ ]:
df_clean = df.copy()

# Age: median
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)

# Country: mode
country_mode = df_clean['Country'].mode()[0]
df_clean['Country'] = df_clean['Country'].fillna(country_mode)

# Gender: mode
gender_mode = df_clean['Gender'].mode()[0]
df_clean['Gender'] = df_clean['Gender'].fillna(gender_mode)

# ADR: median
adr_median = df_clean['ADR'].median()
df_clean['ADR'] = df_clean['ADR'].fillna(adr_median)

# Review_Score: mean
review_mean = df_clean['Review_Score'].mean()
df_clean['Review_Score'] = df_clean['Review_Score'].fillna(review_mean)

# Loyalty_Points: 0
df_clean['Loyalty_Points'] = df_clean['Loyalty_Points'].fillna(0)

# Total_Nights: mode
total_nights_mode = df_clean['Total_Nights'].mode()[0]
df_clean['Total_Nights'] = df_clean['Total_Nights'].fillna(total_nights_mode)

# Room_Type: mode
room_mode = df_clean['Room_Type'].mode()[0]
df_clean['Room_Type'] = df_clean['Room_Type'].fillna(room_mode)

# Market_Segment: mode
market_mode = df_clean['Market_Segment'].mode()[0]
df_clean['Market_Segment'] = df_clean['Market_Segment'].fillna(market_mode)

# Special_Requests: 0
df_clean['Special_Requests'] = df_clean['Special_Requests'].fillna(0)

# Agent_ID: Unknown
df_clean['Agent_ID'] = df_clean['Agent_ID'].fillna('Unknown')

# Meal_Type: mode
meal_mode = df_clean['Meal_Type'].mode()[0]
df_clean['Meal_Type'] = df_clean['Meal_Type'].fillna(meal_mode)

# Num_Guests: median
guests_median = df_clean['Num_Guests'].median()
df_clean['Num_Guests'] = df_clean['Num_Guests'].fillna(guests_median)

# Drop rows with missing critical identifiers
before_drop = len(df_clean)
df_clean.dropna(subset=['customer_id', 'Name', 'email', 'Booking_Date', 'Check_In', 'Check_Out'], inplace=True)
after_drop = len(df_clean)
print(f"Dropped {before_drop - after_drop} rows with missing critical identifiers.")

print("\nMissing values after imputation:")
remaining = df_clean.isnull().sum()
display(remaining[remaining > 0])


## 5. Duplicate Removal

Identify and remove completely duplicated rows to avoid bias in analysis.


In [ ]:
dup_before = df_clean.duplicated().sum()
print(f"Duplicate rows BEFORE removal: {dup_before}")

df_clean.drop_duplicates(inplace=True)

dup_after = df_clean.duplicated().sum()
print(f"Duplicate rows AFTER removal: {dup_after}")
print(f"Total duplicates removed: {dup_before - dup_after}")
print(f"Rows remaining: {len(df_clean)}")


## 6. Standardize Text Data

We clean inconsistent text by:
- Converting to title case
- Trimming whitespace
- Fixing common misspellings and abbreviations


In [ ]:
text_cols = ['Country', 'Gender', 'Guest_Type', 'Meal_Type', 'Market_Segment',
             'Deposit_Type', 'Customer_Type', 'Booking_Status']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        df_clean[col] = df_clean[col].replace('Nan', np.nan)

df_clean['Gender'] = df_clean['Gender'].replace({
    'Male': 'Male', 'M': 'Male', 'Male ': 'Male',
    'Female': 'Female', 'F': 'Female',
    'Mle': 'Male', 'Feale': 'Female'
})

df_clean['Country'] = df_clean['Country'].replace({
    'United States': 'USA', 'U.S.A.': 'USA',
    'United Kingdom': 'United Kingdom',
    'Uk': 'United Kingdom', 'Deu': 'Germany',
    'Portuagal': 'Portugal', 'Unk': 'Unknown'
})

print("Text standardization applied.\n")
for col in text_cols:
    if col in df_clean.columns:
        print(f"\n{col}: {sorted(df_clean[col].dropna().unique())[:15]}")


## 7. Data Type Conversion

Convert columns to their proper types for analysis:
- **Dates:** `object` to `datetime64[ns]`
- **Numerics:** Ensure correct numeric types
- **Binary:** Encode Parking_Space as 0/1


In [ ]:
# Convert datetime columns
for col in ['Booking_Date', 'Check_In', 'Check_Out']:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# Drop rows with invalid dates
invalid_dates = df_clean[df_clean[['Booking_Date', 'Check_In', 'Check_Out']].isna().any(axis=1)]
print(f"Dropped {len(invalid_dates)} rows with invalid or missing dates.")

# Convert numeric columns
numeric_cols_convert = ['Age', 'Num_Guests', 'Total_Nights', 'ADR', 'Total_Revenue',
                        'Lead_Time', 'Previous_Cancellations', 'Previous_Bookings_Not_Canceled',
                        'Review_Score', 'Loyalty_Points']
for col in numeric_cols_convert:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Is_Repeated_Guest to nullable integer
df_clean['Is_Repeated_Guest'] = pd.to_numeric(df_clean['Is_Repeated_Guest'], errors='coerce').astype('Int64')

# Parking_Space to binary
ps_map = {'Yes': 1, 'No': 0, 'Y': 1, 'N': 0}
df_clean['Parking_Space'] = df_clean['Parking_Space'].replace(ps_map)
df_clean['Parking_Space'] = pd.to_numeric(df_clean['Parking_Space'], errors='coerce')

# Special_Requests to int where possible
df_clean['Special_Requests'] = pd.to_numeric(df_clean['Special_Requests'], errors='coerce')

print("\nData types after conversion:")
display(df_clean.dtypes)


## 8. Outlier Detection

We use the **IQR method** (1.5 x IQR rule) to detect and cap outliers in key numerical columns.
This preserves data volume while reducing the impact of extreme values.


In [ ]:
def cap_outliers_iqr(series, column_name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    print(f"  {column_name}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, Lower={lower_bound:.2f}, Upper={upper_bound:.2f}")
    print(f"    Outliers detected: {len(outliers)}")
    return series.clip(lower=lower_bound, upper=upper_bound)

df_clean['Age'] = cap_outliers_iqr(df_clean['Age'], 'Age')
df_clean['ADR'] = cap_outliers_iqr(df_clean['ADR'], 'ADR')
df_clean['Total_Nights'] = cap_outliers_iqr(df_clean['Total_Nights'], 'Total_Nights')
df_clean['Lead_Time'] = cap_outliers_iqr(df_clean['Lead_Time'], 'Lead_Time')
df_clean['Total_Revenue'] = cap_outliers_iqr(df_clean['Total_Revenue'], 'Total_Revenue')
df_clean['Previous_Cancellations'] = cap_outliers_iqr(df_clean['Previous_Cancellations'], 'Previous_Cancellations')
df_clean['Review_Score'] = cap_outliers_iqr(df_clean['Review_Score'], 'Review_Score')
df_clean['Loyalty_Points'] = cap_outliers_iqr(df_clean['Loyalty_Points'], 'Loyalty_Points')

print("\nOutliers capped using IQR method.")


## 9. Invalid Data Handling

Remove logically impossible values:
- Age <= 0 or > 100
- Negative ADR or Total_Nights
- Negative Lead_Time


In [ ]:
invalid_age = len(df_clean[(df_clean['Age'] <= 0) | (df_clean['Age'] > 100)])
print(f"Invalid ages (<=0 or >100): {invalid_age}")
df_clean = df_clean[(df_clean['Age'] > 0) & (df_clean['Age'] <= 100)]

invalid_adr = len(df_clean[df_clean['ADR'] < 0])
print(f"Negative ADR values: {invalid_adr}")
df_clean = df_clean[df_clean['ADR'] >= 0]

invalid_nights = len(df_clean[df_clean['Total_Nights'] < 0])
print(f"Negative Total_Nights: {invalid_nights}")
df_clean = df_clean[df_clean['Total_Nights'] >= 0]

invalid_lead = len(df_clean[df_clean['Lead_Time'] < 0])
print(f"Negative Lead_Time: {invalid_lead}")
df_clean = df_clean[df_clean['Lead_Time'] >= 0]

print(f"Rows after invalid data removal: {len(df_clean)}")


## 10. Rename Columns

Make column names clean and consistent.


In [ ]:
rename_mapping = {
    'customer_id': 'Customer_ID',
    'Name': 'Customer_Name',
    'email': 'Email'
}
df_clean.rename(columns=rename_mapping, inplace=True)

column_order = [
    'Customer_ID', 'Customer_Name', 'Email', 'Country', 'Gender', 'Age',
    'Booking_Date', 'Check_In', 'Check_Out', 'Num_Guests', 'Guest_Type',
    'Meal_Type', 'Room_Type', 'Market_Segment', 'Deposit_Type',
    'Customer_Type', 'Previous_Cancellations', 'Total_Nights', 'ADR',
    'Total_Revenue', 'Lead_Time', 'Parking_Space', 'Special_Requests',
    'Booking_Status', 'Rating', 'Review_Score', 'Loyalty_Points',
    'Is_Repeated_Guest', 'Previous_Bookings_Not_Canceled', 'Agent_ID', 'Company_ID'
]
df_clean = df_clean[column_order]

print("Columns renamed and reordered.")
print(f"New column names:\n{list(df_clean.columns)}")


## 11. Final Dataset Validation

Verify that:
- No duplicates remain
- Unwanted missing values are handled
- Datatypes are correct
- Column names are clean


In [ ]:
print(f"Total duplicates: {df_clean.duplicated().sum()}")
print(f"Total missing values:\n{df_clean.isnull().sum()[df_clean.isnull().sum() > 0]}")
print("\nData types:")
display(df_clean.dtypes)
print("\nColumn name format valid: ", all("".join(c.split("_")).isalnum() for c in df_clean.columns))


## 12. Save Clean Dataset

Export the cleaned dataset to a CSV file.


In [ ]:
cleaned_path = os.path.join(PROJECT_DIR, "cleaned_dataset.csv")
df_clean.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_path}")
print(f"Shape: {df_clean.shape}")


## 13. Final Summary

We summarize the impact of each cleaning step.


In [ ]:
original_rows = 2500
final_rows = len(df_clean)
duplicates_removed = dup_before
invalid_removed = original_rows - final_rows + duplicates_removed

summary_text = f"""
Data Cleaning Summary
=====================
Original Dataset Rows:          {original_rows}
Final Cleaned Dataset Rows:     {final_rows}
Total Rows Removed:             {original_rows - final_rows}
Duplicate Rows Removed:         {duplicates_removed}
Invalid Data Rows Removed:      {invalid_removed - duplicates_removed}

Steps Completed:
  - Missing values handled
  - Duplicates removed
  - Text standardized
  - Data types converted
  - Outliers treated (IQR)
  - Invalid entries removed
  - Columns renamed
"""
print(summary_text)


## Visualizations

### Missing Values After Cleaning


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(df_clean.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap (After Cleaning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "missing_values_after.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: missing_values_after.png")


### Boxplot After Cleaning


In [ ]:
numeric_cols_c = df_clean.select_dtypes(include=[np.number]).columns.tolist()
plot_df_c = df_clean[numeric_cols_c].dropna()
plt.figure(figsize=(14, 8))
plot_df_c.boxplot()
plt.xticks(rotation=45)
plt.title('Boxplot of Numeric Features (After Cleaning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "boxplot_after.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: boxplot_after.png")


### Distribution of Key Numerical Features


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df_clean['Age'].dropna(), kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Age Distribution')

sns.histplot(df_clean['ADR'].dropna(), kde=True, ax=axes[0, 1], color='darkgreen')
axes[0, 1].set_title('ADR Distribution')

sns.histplot(df_clean['Total_Nights'].dropna(), kde=True, ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Total Nights Distribution')

sns.histplot(df_clean['Lead_Time'].dropna(), kde=True, ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Lead Time Distribution')

plt.suptitle('Distribution of Key Numerical Features (After Cleaning)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "distribution_plots.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: distribution_plots.png")


### Correlation Heatmap


In [ ]:
numeric_for_corr = df_clean[['Age', 'Num_Guests', 'Previous_Cancellations', 'Total_Nights',
                              'ADR', 'Total_Revenue', 'Lead_Time', 'Parking_Space',
                              'Special_Requests', 'Review_Score', 'Loyalty_Points',
                              'Is_Repeated_Guest']].dropna()
plt.figure(figsize=(12, 10))
corr_matrix = numeric_for_corr.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap (After Cleaning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "correlation_heatmap.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: correlation_heatmap.png")


### Final Dataset Preview


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('tight')
ax.axis('off')
table_data = df_clean.head(10)[['Customer_ID', 'Customer_Name', 'Country', 'Gender', 'Age',
                                 'Booking_Date', 'Check_In', 'Check_Out', 'Num_Guests',
                                 'Room_Type', 'Total_Nights', 'ADR', 'Booking_Status']].values
col_labels = ['Customer_ID', 'Customer_Name', 'Country', 'Gender', 'Age',
              'Booking_Date', 'Check_In', 'Check_Out', 'Num_Guests',
              'Room_Type', 'Total_Nights', 'ADR', 'Booking_Status']
table = ax.table(cellText=table_data, colLabels=col_labels, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2)
for key, cell in table.get_celld().items():
    if key[0] == 0:
        cell.set_facecolor('#40466e')
        cell.set_text_props(weight='bold', color='white')
plt.title('Final Cleaned Dataset Preview (First 10 Rows)', fontsize=14, fontweight='bold', pad=20)
plt.savefig(os.path.join(IMAGES_DIR, "final_dataset_preview.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: final_dataset_preview.png")
